In [ ]:
import matplotlib
matplotlib.use("MacOSX")
import matplotlib.pyplot as plt

from eeg_denoising import config, data, montage, eeg_utils
from eeg_denoising.eeg_utils import get_segments
from collections import defaultdict
import numpy as np



master = data.load_master()

durations_by_label = defaultdict(list)

for patient_id, patient in data.iter_patients(master):
    for recording_id, recording in patient["recordings"].items():
        for seg in get_segments(recording, include_seiz=True):
            durations_by_label[seg["label"]].append(seg["duration"])

for label, durations in durations_by_label.items():
    print(f"{label:12} count: {len(durations):5}  min: {np.min(durations):.2f}s  max: {np.max(durations):.2f}s  median: {np.median(durations):.2f}s")

In [ ]:
from collections import Counter

label_counts = Counter()

for patient_id, patient in data.iter_patients(master):
    for recording_id, recording in patient["recordings"].items():
        for seg in get_segments(recording, include_seiz=True):
            label_counts[seg["label"]] += 1

for label, count in label_counts.most_common():
    print(f"{label:12} {count}")

In [ ]:
import numpy as np
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import column
from bokeh.models import HoverTool, Div
from eeg_denoising import config, data, montage, eeg_utils
from eeg_denoising.eeg_utils import get_segments


def compute_magnitude(signal, sfreq):
    magnitude = np.abs(np.fft.rfft(signal))
    freqs = np.fft.rfftfreq(len(signal), 1 / sfreq)
    return freqs, magnitude

def compute_phase(signal, sfreq):
    phase = np.angle(np.fft.rfft(signal))
    freqs = np.fft.rfftfreq(len(signal), 1 / sfreq)
    return freqs, phase
master = data.load_master()



# collect one representative segment per label per channel
label_signals = {}

for patient_id, patient in data.iter_patients(master):
    for recording_id, recording in patient["recordings"].items():
        remaining = [seg for seg in get_segments(recording, include_seiz=True)
                     if seg["label"] not in label_signals]
        if not remaining:
            continue

        edf_data, ch_names, sfreq = eeg_utils.load_edf(recording, config.PATH_TO_TUAR_EDF_FILES, montage.TUAR_TCP_AR)

        for seg in remaining:
            label = seg["label"]
            if label not in label_signals:
                label_signals[label] = {}
            if seg["channel"] not in label_signals[label]:
                signal = eeg_utils.extract_signal(edf_data, ch_names, seg, sfreq)
                if len(signal) > int(sfreq * 2):
                    label_signals[label][seg["channel"]] = (signal, sfreq)

        del edf_data

    if len(label_signals) >= 10:
        break

# plot — one html file per label, opens in browser automatically
output_dir = config.PATH_TO_ROOT / "notebooks"

for label, channel_signals in label_signals.items():
    if not channel_signals:
        continue

    output_file(
        filename=str(output_dir / f"magnitude_{label}.html"),
        title=f"Magnitude — {label}"
    )

    plots = []
    plots.append(Div(text=f"<h2>{label}</h2>"))

    for channel, (signal, sfreq) in channel_signals.items():
        freqs, magnitude = compute_magnitude(signal, sfreq)
        magnitude_normalized = magnitude / magnitude.max() * 1e4

        mask = freqs <= 70
        f = freqs[mask]
        m = magnitude_normalized[mask]

        p = figure(
            title=channel,
            width=900,
            height=150,
            x_range=(0, 70),
            y_range=(0, 1e4),
            tools="pan,box_zoom,wheel_zoom,reset,save",
        )

        p.add_tools(HoverTool(tooltips=[("freq", "@x{0.1f} Hz"), ("magnitude", "@y{0.0f}")]))
        p.line(f, m, line_width=0.8, color="steelblue")
        p.xaxis.axis_label = "Frequency (Hz)" if channel == list(channel_signals.keys())[-1] else ""
        p.yaxis.axis_label = "Magnitude"
        p.title.text_font_size = "10px"

        plots.append(p)

    show(column(*plots))

In [ ]:
import numpy as np
from pathlib import Path
from scipy.interpolate import interp1d
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import column
from bokeh.models import HoverTool, Div
from eeg_denoising import config, data, montage, eeg_utils
from eeg_denoising.eeg_utils import get_segments
from collections import defaultdict


def compute_magnitude(signal, sfreq):
    magnitude = np.abs(np.fft.rfft(signal))
    freqs = np.fft.rfftfreq(len(signal), 1 / sfreq)
    return freqs, magnitude


def compute_phase(signal, sfreq):
    phase = np.angle(np.fft.rfft(signal))
    freqs = np.fft.rfftfreq(len(signal), 1 / sfreq)
    return freqs, phase


def resample_to(arr, length):
    x_old = np.linspace(0, 1, len(arr))
    x_new = np.linspace(0, 1, length)
    return interp1d(x_old, arr)(x_new)


FIXED_LEN = 1024

master = data.load_master()
n_patients = len(master["Patients"])

# create output directories
plots_dir        = config.PATH_TO_ROOT / "plots"
trim_dir         = plots_dir / "01_trim_to_minimum"
pad_dir          = plots_dir / "02_zero_pad_to_maximum"
interpolate_dir  = plots_dir / "03_interpolate_to_fixed_length"

for d in [trim_dir, pad_dir, interpolate_dir]:
    d.mkdir(parents=True, exist_ok=True)

# accumulate raw segments per label per channel
label_segments = defaultdict(lambda: defaultdict(list))
label_segment_counts = defaultdict(lambda: defaultdict(int))

for patient_id, patient in data.iter_patients(master):
    for recording_id, recording in patient["recordings"].items():
        if recording["edf"] is None:
            continue
        segments = get_segments(recording, include_seiz=True)
        if not segments:
            continue

        edf_data, ch_names, sfreq = eeg_utils.load_edf(recording, config.PATH_TO_TUAR_EDF_FILES, montage.TUAR_TCP_AR)

        for seg in segments:
            label   = seg["label"]
            channel = seg["channel"]
            signal  = eeg_utils.extract_signal(edf_data, ch_names, seg, sfreq)

            if len(signal) < int(sfreq * 2):
                continue

            _, magnitude = compute_magnitude(signal, sfreq)
            _, phase     = compute_phase(signal, sfreq)

            label_segments[label][channel].append({
                "magnitude": magnitude / magnitude.max(),
                "phase":     phase,
            })
            label_segment_counts[label][channel] += 1

        del edf_data

def plot_spectra(avg_magnitude, avg_phase, freqs, label, channel_data,
                 output_dir, approach_name, approach_description):

    mask = freqs <= 70
    f    = freqs[mask]
    n_segs = sum(label_segment_counts[label].values())

    # magnitude
    output_file(
        filename=str(output_dir / f"magnitude_avg_{label}_{n_segs}segments.html"),
        title=f"[{approach_name}] Average Magnitude — {label}"
    )
    plots = [Div(text=f"<h2>Average Magnitude — {label}</h2>"
                      f"<p><b>Approach:</b> {approach_name} — {approach_description}</p>"
                      f"<p>{n_segs} segments averaged across {n_patients} patients</p>")]

    for channel, mag in avg_magnitude.items():
        n_ch_segs = label_segment_counts[label][channel]

        # compute mask specific to this mag array length
        ch_freqs = np.linspace(0, sfreq / 2, len(mag))
        ch_mask  = ch_freqs <= 70

        p = figure(
            title=f"{channel}  ({n_ch_segs} segments)",
            width=900, height=150,
            x_range=(0, 70), y_range=(0, 1e4),
            tools="pan,box_zoom,wheel_zoom,reset,save",
        )
        p.add_tools(HoverTool(tooltips=[("freq", "@x{0.1f} Hz"), ("magnitude", "@y{0.0f}")]))
        p.line(ch_freqs[ch_mask], mag[ch_mask], line_width=0.8, color="steelblue")
        p.xaxis.axis_label = "Frequency (Hz)" if channel == list(avg_magnitude.keys())[-1] else ""
        p.yaxis.axis_label = "Magnitude"
        p.title.text_font_size = "10px"
        plots.append(p)
    show(column(*plots))

    # phase
    output_file(
        filename=str(output_dir / f"phase_avg_{label}_{n_segs}segments.html"),
        title=f"[{approach_name}] Average Phase — {label}"
    )
    plots = [Div(text=f"<h2>Average Phase — {label}</h2>"
                      f"<p><b>Approach:</b> {approach_name} — {approach_description}</p>"
                      f"<p>{n_segs} segments averaged across {n_patients} patients</p>")]

    for channel, ph in avg_phase.items():
        n_ch_segs = label_segment_counts[label][channel]

        ch_freqs = np.linspace(0, sfreq / 2, len(ph))
        ch_mask  = ch_freqs <= 70

        p = figure(
            title=f"{channel}  ({n_ch_segs} segments)",
            width=900, height=150,
            x_range=(0, 70), y_range=(-np.pi, np.pi),
            tools="pan,box_zoom,wheel_zoom,reset,save",
        )
        p.add_tools(HoverTool(tooltips=[("freq", "@x{0.1f} Hz"), ("phase", "@y{0.3f} rad")]))
        p.line(ch_freqs[ch_mask], ph[ch_mask], line_width=0.8, color="tomato")
        p.xaxis.axis_label = "Frequency (Hz)" if channel == list(avg_phase.keys())[-1] else ""
        p.yaxis.axis_label = "Phase (rad)"
        p.title.text_font_size = "10px"
        plots.append(p)
    show(column(*plots))

for label, channel_data in label_segments.items():
    if not channel_data:
        continue

    mag_trim, phase_trim         = {}, {}
    mag_pad, phase_pad           = {}, {}
    mag_interp, phase_interp     = {}, {}

    for channel, segs in channel_data.items():
        mags   = [s["magnitude"] for s in segs]
        phases = [s["phase"]     for s in segs]

        # ── approach 1: trim to minimum length ───────────────────────────────
        min_len      = min(len(s) for s in mags)
        stacked_mag  = np.array([s[:min_len] for s in mags])
        stacked_phase= np.array([s[:min_len] for s in phases])
        mag_trim[channel]   = np.mean(stacked_mag,   axis=0) * 1e4
        phase_trim[channel] = np.mean(stacked_phase, axis=0)

        # ── approach 2: zero pad to maximum length ────────────────────────────
        max_len      = max(len(s) for s in mags)
        stacked_mag  = np.array([np.pad(s, (0, max_len - len(s))) for s in mags])
        stacked_phase= np.array([np.pad(s, (0, max_len - len(s))) for s in phases])
        mag_pad[channel]   = np.mean(stacked_mag,   axis=0) * 1e4
        phase_pad[channel] = np.mean(stacked_phase, axis=0)

        # ── approach 3: interpolate to fixed length ───────────────────────────
        stacked_mag  = np.array([resample_to(s, FIXED_LEN) for s in mags])
        stacked_phase= np.array([resample_to(s, FIXED_LEN) for s in phases])
        mag_interp[channel]   = np.mean(stacked_mag,   axis=0) * 1e4
        phase_interp[channel] = np.mean(stacked_phase, axis=0)

    # freqs for each approach
    freqs_trim   = np.fft.rfftfreq(min_len,    1 / sfreq)
    freqs_pad    = np.fft.rfftfreq(max_len,    1 / sfreq)
    freqs_interp = np.fft.rfftfreq(FIXED_LEN,  1 / sfreq)

    plot_spectra(mag_trim,   phase_trim,   freqs_trim,   label, channel_data, trim_dir,
                 "Trim to Minimum", "All segments trimmed to the shortest segment length")

    plot_spectra(mag_pad,    phase_pad,    freqs_pad,    label, channel_data, pad_dir,
                 "Zero Pad to Maximum", "Shorter segments zero padded to the longest segment length")

    plot_spectra(mag_interp, phase_interp, freqs_interp, label, channel_data, interpolate_dir,
                 "Interpolate to Fixed Length", f"All segments resampled to {FIXED_LEN} points via cubic interpolation")

In [ ]:
import numpy as np
from pathlib import Path
from scipy.interpolate import interp1d
from bokeh.plotting import figure, show, output_file
from bokeh.layouts import column
from bokeh.models import HoverTool, Div
from eeg_denoising import config, data, montage, eeg_utils
from eeg_denoising.eeg_utils import get_segments
from collections import defaultdict


def compute_frequency_response(signal, sfreq):
    """
    Compute the frequency response in dB.
    Unlike magnitude which is linear, dB scale better represents
    the dynamic range of EEG signals across frequency bands.
    """
    fft_vals = np.fft.rfft(signal)
    freqs    = np.fft.rfftfreq(len(signal), 1 / sfreq)
    response = 20 * np.log10(np.abs(fft_vals) + 1e-10)   # +epsilon avoids log(0)
    return freqs, response


def resample_to(arr, length):
    x_old = np.linspace(0, 1, len(arr))
    x_new = np.linspace(0, 1, length)
    return interp1d(x_old, arr)(x_new)


FIXED_LEN = 1024

master     = data.load_master()
n_patients = len(master["Patients"])

# create output directories
plots_dir       = config.PATH_TO_ROOT / "plots"
freq_trim_dir   = plots_dir / "01_trim_to_minimum"        / "frequency_response"
freq_pad_dir    = plots_dir / "02_zero_pad_to_maximum"    / "frequency_response"
freq_interp_dir = plots_dir / "03_interpolate_to_fixed_length" / "frequency_response"

for d in [freq_trim_dir, freq_pad_dir, freq_interp_dir]:
    d.mkdir(parents=True, exist_ok=True)

# accumulate raw frequency responses per label per channel
label_freq_accum     = defaultdict(lambda: defaultdict(list))
label_segment_counts = defaultdict(lambda: defaultdict(int))

for patient_id, patient in data.iter_patients(master):
    for recording_id, recording in patient["recordings"].items():
        if recording["edf"] is None:
            continue
        segments = get_segments(recording, include_seiz=True)
        if not segments:
            continue

        edf_data, ch_names, sfreq = eeg_utils.load_edf(
            recording, config.PATH_TO_TUAR_EDF_FILES, montage.TUAR_TCP_AR
        )

        for seg in segments:
            label   = seg["label"]
            channel = seg["channel"]
            signal  = eeg_utils.extract_signal(edf_data, ch_names, seg, sfreq)

            if len(signal) < int(sfreq * 2):
                continue

            _, response = compute_frequency_response(signal, sfreq)

            # normalise to 0dB peak before accumulating
            response_norm = response - response.max()
            label_freq_accum[label][channel].append(response_norm)
            label_segment_counts[label][channel] += 1

        del edf_data


def plot_frequency_response(avg_response, label, output_dir, approach_name, approach_description):
    n_segs = sum(label_segment_counts[label].values())

    output_file(
        filename=str(output_dir / f"freq_response_avg_{label}_{n_segs}segments.html"),
        title=f"[{approach_name}] Average Frequency Response — {label}"
    )

    plots = [Div(text=f"<h2>Average Frequency Response (dB) — {label}</h2>"
                      f"<p><b>Approach:</b> {approach_name} — {approach_description}</p>"
                      f"<p>{n_segs} segments averaged across {n_patients} patients | "
                      f"Normalised to 0 dB peak</p>")]

    for channel, resp in avg_response.items():
        n_ch_segs = label_segment_counts[label][channel]

        ch_freqs = np.linspace(0, sfreq / 2, len(resp))
        ch_mask  = ch_freqs <= 70

        p = figure(
            title=f"{channel}  ({n_ch_segs} segments)",
            width=900,
            height=150,
            x_range=(0, 70),
            y_range=(-60, 0),   # dB range — 0 is peak, -60 is noise floor
            tools="pan,box_zoom,wheel_zoom,reset,save",
        )
        p.add_tools(HoverTool(tooltips=[
            ("freq",     "@x{0.1f} Hz"),
            ("response", "@y{0.1f} dB")
        ]))
        p.line(ch_freqs[ch_mask], resp[ch_mask], line_width=0.8, color="mediumseagreen")
        p.xaxis.axis_label = "Frequency (Hz)" if channel == list(avg_response.keys())[-1] else ""
        p.yaxis.axis_label = "Response (dB)"
        p.title.text_font_size = "10px"
        plots.append(p)

    show(column(*plots))


for label, channel_data in label_freq_accum.items():
    if not channel_data:
        continue

    freq_trim   = {}
    freq_pad    = {}
    freq_interp = {}

    for channel, responses in channel_data.items():

        # ── approach 1: trim to minimum length ───────────────────────────────
        min_len  = min(len(r) for r in responses)
        stacked  = np.array([r[:min_len] for r in responses])
        freq_trim[channel] = np.mean(stacked, axis=0)

        # ── approach 2: zero pad to maximum length ────────────────────────────
        max_len  = max(len(r) for r in responses)
        stacked  = np.array([np.pad(r, (0, max_len - len(r))) for r in responses])
        freq_pad[channel] = np.mean(stacked, axis=0)

        # ── approach 3: interpolate to fixed length ───────────────────────────
        stacked  = np.array([resample_to(r, FIXED_LEN) for r in responses])
        freq_interp[channel] = np.mean(stacked, axis=0)

    plot_frequency_response(
        freq_trim, label, freq_trim_dir,
        "Trim to Minimum",
        "All segments trimmed to the shortest segment length"
    )

    plot_frequency_response(
        freq_pad, label, freq_pad_dir,
        "Zero Pad to Maximum",
        "Shorter segments zero padded to the longest segment length"
    )

    plot_frequency_response(
        freq_interp, label, freq_interp_dir,
        "Interpolate to Fixed Length",
        f"All segments resampled to {FIXED_LEN} points via cubic interpolation"
    )